In [18]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

# Funciona se o notebook estiver na raiz ou em notebooks/
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Não foi possível localizar a raiz do projeto.")

sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Luan\Desktop\Data Science


In [19]:
import pandas as pd

from src.features.build_features import (
    ChurnFeatureEngineer,
    build_features,
    get_features_for_modeling,
)
from src.features.feature_contract import (
    RAW_FEATURES,
    MODEL_FEATURES,
    TARGET_COLUMN,
)

print("Entradas brutas:", RAW_FEATURES)
print("Features internas:", MODEL_FEATURES)
print("Target:", TARGET_COLUMN)

Entradas brutas: ('Age', 'Tenure', 'NumOfProducts')
Features internas: ('NumOfProducts', 'Age_Squared', 'Age_Tenure_Interaction')
Target: Exited


In [20]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Customer-Churn-Records.csv"

df = pd.read_csv(DATA_PATH)

print("Dimensão:", df.shape)
display(df.head())

Dimensão: (10000, 18)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


In [21]:
X, y = get_features_for_modeling(df)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Colunas de X:", X.columns.tolist())
print("Distribuição do target:")
display(y.value_counts(normalize=True).rename("proporção"))

display(X.head())

X shape: (10000, 3)
y shape: (10000,)
Colunas de X: ['Age', 'Tenure', 'NumOfProducts']
Distribuição do target:


Exited
0    0.7962
1    0.2038
Name: proporção, dtype: float64

,Age,Tenure,NumOfProducts
0,42,2,1
1,41,1,1
2,42,8,3
3,39,1,2
4,43,2,1


In [22]:
engineered = build_features(X)

print("Dimensão:", engineered.shape)
print("Features geradas:", engineered.columns.tolist())

display(engineered.head(10))

Dimensão: (10000, 3)
Features geradas: ['NumOfProducts', 'Age_Squared', 'Age_Tenure_Interaction']


,NumOfProducts,Age_Squared,Age_Tenure_Interaction
0,1,1764,84
1,1,1681,41
2,3,1764,336
3,2,1521,39
4,1,1849,86
5,2,1936,352
6,2,2500,350
7,4,841,116
8,2,1936,176
9,1,729,54


In [23]:
comparison = pd.DataFrame({
    "Age": X["Age"],
    "Tenure": X["Tenure"],
    "NumOfProducts": X["NumOfProducts"],
    "Age_Squared_esperado": X["Age"] ** 2,
    "Age_Squared_obtido": engineered["Age_Squared"],
    "Interaction_esperada": X["Age"] * X["Tenure"],
    "Interaction_obtida": engineered["Age_Tenure_Interaction"],
})

display(comparison.head(10))

,Age,Tenure,NumOfProducts,Age_Squared_esperado,Age_Squared_obtido,Interaction_esperada,Interaction_obtida
0,42,2,1,1764,1764,84,84
1,41,1,1,1681,1681,41,41
2,42,8,3,1764,1764,336,336
3,39,1,2,1521,1521,39,39
4,43,2,1,1849,1849,86,86
5,44,8,2,1936,1936,352,352
6,50,7,2,2500,2500,350,350
7,29,4,4,841,841,116,116
8,44,4,2,1936,1936,176,176
9,27,2,1,729,729,54,54


In [24]:
assert list(X.columns) == list(RAW_FEATURES)
assert list(engineered.columns) == list(MODEL_FEATURES)
assert len(engineered) == len(X)

pd.testing.assert_series_equal(
    engineered["Age_Squared"],
    X["Age"] ** 2,
    check_names=False,
)

pd.testing.assert_series_equal(
    engineered["Age_Tenure_Interaction"],
    X["Age"] * X["Tenure"],
    check_names=False,
)

pd.testing.assert_series_equal(
    engineered["NumOfProducts"],
    X["NumOfProducts"],
    check_names=False,
)

print("Todas as validações passaram.")

Todas as validações passaram.


In [25]:
transformer = ChurnFeatureEngineer()

transformer.fit(X)
transformed = transformer.transform(X)

pd.testing.assert_frame_equal(engineered, transformed)

print("build_features() e ChurnFeatureEngineer produzem o mesmo resultado.")
display(transformed.head())

build_features() e ChurnFeatureEngineer produzem o mesmo resultado.


,NumOfProducts,Age_Squared,Age_Tenure_Interaction
0,1,1764,84
1,1,1681,41
2,3,1764,336
3,2,1521,39
4,1,1849,86


In [26]:
sample = pd.DataFrame({
    "Age": [30, 45, 60],
    "Tenure": [5, 10, 8],
    "NumOfProducts": [1, 2, 3],
})

sample_result = build_features(sample)

display(pd.concat(
    [sample.add_prefix("raw_"), sample_result.add_prefix("feature_")],
    axis=1,
))

,raw_Age,raw_Tenure,raw_NumOfProducts,feature_NumOfProducts,feature_Age_Squared,feature_Age_Tenure_Interaction
0,30,5,1,1,900,150
1,45,10,2,2,2025,450
2,60,8,3,3,3600,480


In [27]:
# Coluna ausente
try:
    build_features(sample.drop(columns=["Age"]))
except ValueError as exc:
    print("Erro esperado para coluna ausente:")
    print(exc)

# Tipo inválido
invalid_sample = sample.copy()
invalid_sample["Age"] = ["trinta", "quarenta", "sessenta"]

try:
    build_features(invalid_sample)
except ValueError as exc:
    print("\nErro esperado para tipo não numérico:")
    print(exc)

Erro esperado para coluna ausente:
Missing raw input columns: ['Age']

Erro esperado para tipo não numérico:
Raw input columns must be numeric: ['Age']


In [28]:
from src.pipelines.inference_pipeline import InferencePipeline

v2 = InferencePipeline("v2")
v3 = InferencePipeline("v3")
v4 = InferencePipeline("v4")

for pipeline in (v2, v3, v4):
    print({
        "version": pipeline.version,
        "threshold": pipeline.threshold,
        "features": pipeline.feature_names,
        "strict_contract": pipeline.strict_input_contract,
    })

{'version': 'v2', 'threshold': 0.5, 'features': ['NumOfProducts', 'Age_Squared', 'Age_Tenure_Interaction'], 'strict_contract': False}
{'version': 'v3', 'threshold': 0.34, 'features': ['NumOfProducts', 'Age_Squared', 'Age_Tenure_Interaction'], 'strict_contract': False}
{'version': 'v4', 'threshold': 0.33, 'features': ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography', 'Gender'], 'strict_contract': True}


In [29]:
def prediction_dataframe(pipeline, input_df):
    result = pipeline.predict_with_confidence(input_df)

    return pd.DataFrame({
        "model": pipeline.version,
        "prediction": result["predictions"],
        "probability": result["probabilities"],
        "high_confidence": result["high_confidence"],
        "threshold": pipeline.threshold,
    }, index=input_df.index)


v2_predictions = prediction_dataframe(v2, sample)
v3_predictions = prediction_dataframe(v3, sample)

comparison_v2_v3 = pd.concat(
    [
        sample.reset_index(drop=True),
        v2_predictions.add_prefix("v2_").reset_index(drop=True),
        v3_predictions.add_prefix("v3_").reset_index(drop=True),
    ],
    axis=1,
)

display(comparison_v2_v3)

,Age,Tenure,NumOfProducts,v2_model,v2_prediction,v2_probability,v2_high_confidence,v2_threshold,v3_model,v3_prediction,v3_probability,v3_high_confidence,v3_threshold
0,30,5,1,v2,0,0.139729,True,0.5,v3,0,0.121160,True,0.34
1,45,10,2,v2,0,0.112127,True,0.5,v3,0,0.176373,True,0.34
2,60,8,3,v2,1,0.967748,True,0.5,v3,1,0.935668,True,0.34


In [30]:
profiles = ["default", "balanced", "high_recall", "high_precision", "lowest_cost"]

profile_results = []

for version in ("v3", "v4"):
    for profile in profiles:
        pipeline = InferencePipeline(version, threshold_profile=profile)

        input_data = sample if version == "v3" else None

        profile_results.append({
            "model": version,
            "profile": profile,
            "threshold": pipeline.threshold,
        })

display(pd.DataFrame(profile_results))

,model,profile,threshold
0,v3,default,0.50
1,v3,balanced,0.34
2,v3,high_recall,0.21
3,v3,high_precision,0.54
4,v3,lowest_cost,0.17
5,v4,default,0.50
6,v4,balanced,0.33
7,v4,high_recall,0.18
8,v4,high_precision,0.51
9,v4,lowest_cost,0.17


In [31]:
from src.pipelines.feature_expansion_pipeline import prepare_bank_data

bank_df = prepare_bank_data(df)

v4_input = bank_df.loc[:, v4.required_input_features].head(10).copy()

print("Features obrigatórias da v4:")
print(v4.required_input_features)

display(v4_input)

Features obrigatórias da v4:
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography', 'Gender']


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography,Gender
0,619,42,2,0.00,1,1,1,101348.88,France,Female
1,608,41,1,83807.86,1,0,1,112542.58,Spain,Female
2,502,42,8,159660.80,3,1,0,113931.57,France,Female
3,699,39,1,0.00,2,0,0,93826.63,France,Female
4,850,43,2,125510.82,1,1,1,79084.10,Spain,Female
5,645,44,8,113755.78,2,1,0,149756.71,Spain,Male
6,822,50,7,0.00,2,1,1,10062.80,France,Male
7,376,29,4,115046.74,4,1,0,119346.88,Germany,Female
8,501,44,4,142051.07,2,0,1,74940.50,France,Male
9,684,27,2,134603.88,1,1,1,71725.73,France,Male


In [32]:
v4_predictions = prediction_dataframe(v4, v4_input)

v4_results = pd.concat(
    [
        v4_input.reset_index(drop=True),
        v4_predictions.reset_index(drop=True),
    ],
    axis=1,
)

display(v4_results)

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography,Gender,model,prediction,probability,high_confidence,threshold
0,619,42,2,0.00,1,1,1,101348.88,France,Female,v4,0,0.327281,False,0.33
1,608,41,1,83807.86,1,0,1,112542.58,Spain,Female,v4,0,0.186005,True,0.33
2,502,42,8,159660.80,3,1,0,113931.57,France,Female,v4,1,0.923739,True,0.33
3,699,39,1,0.00,2,0,0,93826.63,France,Female,v4,0,0.049242,True,0.33
4,850,43,2,125510.82,1,1,1,79084.10,Spain,Female,v4,0,0.126546,True,0.33
5,645,44,8,113755.78,2,1,0,149756.71,Spain,Male,v4,0,0.237193,False,0.33
6,822,50,7,0.00,2,1,1,10062.80,France,Male,v4,0,0.058381,True,0.33
7,376,29,4,115046.74,4,1,0,119346.88,Germany,Female,v4,1,0.962667,True,0.33
8,501,44,4,142051.07,2,0,1,74940.50,France,Male,v4,0,0.140262,True,0.33
9,684,27,2,134603.88,1,1,1,71725.73,France,Male,v4,0,0.045671,True,0.33


In [33]:
v4_predictions = prediction_dataframe(v4, v4_input)

v4_results = pd.concat(
    [
        v4_input.reset_index(drop=True),
        v4_predictions.reset_index(drop=True),
    ],
    axis=1,
)

display(v4_results)

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography,Gender,model,prediction,probability,high_confidence,threshold
0,619,42,2,0.00,1,1,1,101348.88,France,Female,v4,0,0.327281,False,0.33
1,608,41,1,83807.86,1,0,1,112542.58,Spain,Female,v4,0,0.186005,True,0.33
2,502,42,8,159660.80,3,1,0,113931.57,France,Female,v4,1,0.923739,True,0.33
3,699,39,1,0.00,2,0,0,93826.63,France,Female,v4,0,0.049242,True,0.33
4,850,43,2,125510.82,1,1,1,79084.10,Spain,Female,v4,0,0.126546,True,0.33
5,645,44,8,113755.78,2,1,0,149756.71,Spain,Male,v4,0,0.237193,False,0.33
6,822,50,7,0.00,2,1,1,10062.80,France,Male,v4,0,0.058381,True,0.33
7,376,29,4,115046.74,4,1,0,119346.88,Germany,Female,v4,1,0.962667,True,0.33
8,501,44,4,142051.07,2,0,1,74940.50,France,Male,v4,0,0.140262,True,0.33
9,684,27,2,134603.88,1,1,1,71725.73,France,Male,v4,0,0.045671,True,0.33


## Teste de um cliente manual na v4

In [34]:
bank_customer = pd.DataFrame([{
    "CreditScore": 650,
    "Age": 45,
    "Tenure": 5,
    "Balance": 100_000.0,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 75_000.0,
    "Geography": "France",
    "Gender": "Female",
}])

manual_result = v4.predict_with_confidence(bank_customer)

print("Classe:", manual_result["predictions"][0])
print("Probabilidade de churn:", f"{manual_result['probabilities'][0]:.2%}")
print("Alta confiança:", manual_result["high_confidence"][0])
print("Threshold:", v4.threshold)

Classe: 0
Probabilidade de churn: 21.61%
Alta confiança: False
Threshold: 0.33


## Validação do contrato estrito da v4

In [35]:
incomplete_customer = pd.DataFrame([{
    "Age": 45,
    "Tenure": 5,
    "NumOfProducts": 2,
}])

try:
    v4.predict_with_confidence(incomplete_customer)
    raise AssertionError("A v4 deveria rejeitar o payload incompleto")
except ValueError as exc:
    print("Erro esperado:")
    print(exc)

Erro esperado:
Incomplete v4 input; missing columns=['CreditScore', 'Balance', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography', 'Gender'], null columns=[]


## Invariância entre previsão individual e batch

In [36]:
import math

invariance_inputs = [
    (v2, X.head(10)),
    (v3, X.head(10)),
    (v4, bank_df.loc[:, v4.required_input_features].head(10)),
]

for pipeline, input_data in invariance_inputs:
    single_probability = pipeline.predict_with_confidence(
        input_data.iloc[[0]]
    )["probabilities"][0]
    batch_probability = pipeline.predict_with_confidence(
        input_data
    )["probabilities"][0]

    assert math.isclose(single_probability, batch_probability, abs_tol=1e-12)
    print(pipeline.version, "single =", single_probability, "batch =", batch_probability)

print("Invariância individual/batch confirmada.")

v2 single = 0.3316822188522505 batch = 0.3316822188522505
v3 single = 0.2774711761841881 batch = 0.2774711761841881
v4 single = 0.3272811309800502 batch = 0.3272811309800502
Invariância individual/batch confirmada.


## Reprodução do holdout e comparação v2, v3 e v4

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)

_, test_indices = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df[TARGET_COLUMN]
)
y_test = df.loc[test_indices, TARGET_COLUMN]

evaluation_inputs = {
    "v2": (v2, df.loc[test_indices, list(RAW_FEATURES)]),
    "v3": (v3, df.loc[test_indices, list(RAW_FEATURES)]),
    "v4": (v4, bank_df.loc[test_indices, v4.required_input_features]),
}

evaluation_rows = []
confusion_matrices = {}

for version, (pipeline, input_data) in evaluation_inputs.items():
    result = pipeline.predict_with_confidence(input_data)
    probabilities = result["probabilities"]
    predictions = result["predictions"]
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()

    evaluation_rows.append({
        "model": version, "threshold": pipeline.threshold,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "pr_auc": average_precision_score(y_test, probabilities),
        "brier_score": brier_score_loss(y_test, probabilities),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    })
    confusion_matrices[version] = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Real: Stay", "Real: Churn"],
        columns=["Predito: Stay", "Predito: Churn"],
    )

evaluation = pd.DataFrame(evaluation_rows).set_index("model")
display(evaluation.style.format({
    "threshold": "{:.2f}", "accuracy": "{:.4f}",
    "precision": "{:.4f}", "recall": "{:.4f}",
    "f1": "{:.4f}", "roc_auc": "{:.4f}",
    "pr_auc": "{:.4f}", "brier_score": "{:.4f}",
}))

,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,brier_score,tn,fp,fn,tp
model,,,,,,,,,,,,
v2,0.50,0.8310,0.6224,0.4363,0.5130,0.8191,0.5969,0.1198,1484,108,230,178
v3,0.34,0.8300,0.5872,0.5613,0.5739,0.8297,0.6199,0.1161,1431,161,179,229
v4,0.33,0.8515,0.6312,0.6544,0.6426,0.8783,0.7310,0.0972,1436,156,141,267


In [38]:
for version, matrix in confusion_matrices.items():
    print(f"\nMatriz de confusão — {version}")
    display(matrix)

expected_v4 = {
    "threshold": 0.33, "precision": 0.6312, "recall": 0.6544,
    "f1": 0.6426, "roc_auc": 0.8783, "pr_auc": 0.7310,
    "brier_score": 0.0972, "tn": 1436, "fp": 156,
    "fn": 141, "tp": 267,
}
print("Referência esperada para a v4:", expected_v4)


Matriz de confusão — v2


,Predito: Stay,Predito: Churn
Real: Stay,1484,108
Real: Churn,230,178



Matriz de confusão — v3


,Predito: Stay,Predito: Churn
Real: Stay,1431,161
Real: Churn,179,229



Matriz de confusão — v4


,Predito: Stay,Predito: Churn
Real: Stay,1436,156
Real: Churn,141,267


Referência esperada para a v4: {'threshold': 0.33, 'precision': 0.6312, 'recall': 0.6544, 'f1': 0.6426, 'roc_auc': 0.8783, 'pr_auc': 0.731, 'brier_score': 0.0972, 'tn': 1436, 'fp': 156, 'fn': 141, 'tp': 267}
